# Lab 4 — Camera Calibration

📷 Camera calibration is the process of estimating the parameters that describe how a real camera forms an image.  
In practice, calibration allows us to model the camera optics, correct lens distortion, and relate **3D points in the world** to their **2D projections in the image**.

In a stereo setup, calibration is even more important because it also estimates the geometric relationship between the **left** and **right** cameras.  
This is the key step that makes it possible to perform **rectification**, compute **disparity**, and later estimate **depth**.

This notebook covers a complete **stereo camera calibration** pipeline with **OpenCV**.

We will work with stereo image pairs acquired from a **side-by-side stereo camera** such as the **ZED family**.  
The goal is to estimate:

1. **Intrinsic parameters** of the left camera  
2. **Intrinsic parameters** of the right camera  
3. **Extrinsic parameters** between the two cameras  
4. **Rectification maps** for epipolar alignment  
5. A compact summary of the calibration quality  

**Important note:** in this notebook we will use the calibration video provided inside the `material` folder.  
The stereo image pairs will be extracted from that video during the lab.

For live acquisition from the students' PCs, use the companion script `Lab4_Camera_Calibration_Local.py`.



## 📦 Requirements and environment setup

We use a standard set of libraries:
- **OpenCV** for chessboard detection and calibration
- **NumPy** for numeric operations
- **Matplotlib** for visualization
- **Pathlib / JSON** for simple file management and summaries

If needed, install them with the following command.


In [ ]:
!pip install opencv-python numpy matplotlib


First, we import the necessary libraries and define the main project directories.

In this lab, we will **not** start from pre-separated stereo image pairs.  
Instead, we will use a **single side-by-side stereo video** provided by the instructor, stored inside the `material` folder.

Each frame of the video contains:

- the **left camera image** on the left half  
- the **right camera image** on the right half  


In [ ]:

# Imports
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)

ROOT_DIR = Path(".")
MATERIAL_DIR = ROOT_DIR / "material"
LEFT_DIR = MATERIAL_DIR / "LEFT_LENS"
RIGHT_DIR = MATERIAL_DIR / "RIGHT_LENS"
SAVES_DIR = ROOT_DIR / "saves"


- **`show_gray()`** — Display a grayscale image with Matplotlib  
- **`show_bgr()`** — Display a color image after converting it from OpenCV BGR format to RGB  
- **`plot_bgr_list()`** — Display a list of color images side by side for quick visual comparison  
- **`draw_horizontal_lines()`** — Overlay horizontal guide lines on an image to visually inspect stereo rectification

In [ ]:

def show_gray(img_gray, title=None, figsize=(8, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(img_gray, cmap="gray")
    if title is not None:
        plt.title(title)
    plt.axis("off")
    plt.show()


def show_bgr(img_bgr, title=None, figsize=(8, 5)):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    if title is not None:
        plt.title(title)
    plt.axis("off")
    plt.show()


def plot_bgr_list(images, titles=None, figsize=(16, 4)):
    n = len(images)
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        plt.subplot(1, n, i + 1)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img_rgb)
        if titles is not None:
            plt.title(titles[i])
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def draw_horizontal_lines(img, step=40, color=(0, 255, 0)):
    out = img.copy()
    h, w = out.shape[:2]
    for y in range(step, h, step):
        cv2.line(out, (0, y), (w, y), color, 1)
    return out


## 🧩 1) Define the calibration dataset and chessboard geometry

Before starting the calibration, we must define the geometry of the chessboard used in the experiment.

In this lab, we use a chessboard with:

- **Square size = 25 mm**
- **Board cols = 9**
- **Board rows = 6**

These values refer to the number of **inner corners**, which is the convention used by OpenCV in `findChessboardCorners()`.

This means that the **printed chessboard** actually has:

- **10 squares horizontally**
- **7 squares vertically**

So, even if the physical board is **10x7 squares**, in OpenCV we must pass the chessboard size as **(9, 6)**.

Why is this important?  
Because calibration requires a set of **known 3D reference points**. The chessboard provides these points in its own reference system: all corners lie on the same plane, so their coordinates can be described as:

- `(0, 0, 0)`
- `(25, 0, 0)`
- `(50, 0, 0)`
- ...
- `(X, Y, 0)`

where the spacing between adjacent corners is exactly the **square size**, i.e. **25 mm**.

In other words, the chessboard gives us a planar set of known 3D points, and OpenCV matches them with the corresponding **2D image points** detected in the left and right images.  
This is the key idea behind camera calibration.

### Chessboard used in this lab

<div align="center">
    <img src="Images4Notebook\chessboard.png" width="20%">
</div>

📌 Make sure that:

- the chessboard definition in the notebook matches the one used in the video
- the square size is expressed in a real metric unit
- the dimensions passed to OpenCV are the **inner corners**, not the number of squares

## 🎞️ 2) Load the stereo video and extract left/right image pairs

In this section, we load the side-by-side stereo video provided for the lab.

Each frame contains:
- the **left camera view** in the left half
- the **right camera view** in the right half

We first display a few sample frames from the original video, then we split each frame into its left and right components and save them into separate folders.

This gives us a stereo image dataset that can be used for checkerboard detection and calibration.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Video path ---
VIDEO_PATH = MATERIAL_DIR / "calibration_video.mp4"

# Create output folders if they do not exist
LEFT_DIR.mkdir(parents=True, exist_ok=True)
RIGHT_DIR.mkdir(parents=True, exist_ok=True)

print("Video path:", VIDEO_PATH)
print("Video exists:", VIDEO_PATH.exists())


def open_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    return cap


def split_stereo_frame(frame):
    h, w = frame.shape[:2]
    mid = w // 2
    left = frame[:, :mid].copy()
    right = frame[:, mid:].copy()
    return left, right


def show_sample_frames(video_path, num_samples=4):
    cap = open_video(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        raise ValueError("The video contains no frames.")

    sample_indices = np.linspace(0, total_frames - 1, num_samples, dtype=int)

    frames_to_plot = []
    titles = []

    for idx in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames_to_plot.append(frame_rgb)
        titles.append(f"Frame {idx}")

    cap.release()

    plt.figure(figsize=(16, 4))
    for i, img in enumerate(frames_to_plot):
        plt.subplot(1, len(frames_to_plot), i + 1)
        plt.imshow(img)
        plt.title(titles[i])
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def extract_and_save_stereo_pairs(video_path, left_dir, right_dir, step=15, max_pairs=None):
    cap = open_video(video_path)

    frame_idx = 0
    saved_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % step == 0:
            left, right = split_stereo_frame(frame)

            left_path = left_dir / f"left_{saved_idx:04d}.png"
            right_path = right_dir / f"right_{saved_idx:04d}.png"

            cv2.imwrite(str(left_path), left)
            cv2.imwrite(str(right_path), right)

            saved_idx += 1

            if max_pairs is not None and saved_idx >= max_pairs:
                break

        frame_idx += 1

    cap.release()
    print(f"Total frames processed: {frame_idx}")
    print(f"Stereo pairs saved: {saved_idx}")
    print(f"Left folder:  {left_dir}")
    print(f"Right folder: {right_dir}")


# 1) Display a few sample frames from the original stereo video
show_sample_frames(VIDEO_PATH, num_samples=4)

# 2) Extract stereo pairs every N frames and save them
extract_and_save_stereo_pairs(
    VIDEO_PATH,
    LEFT_DIR,
    RIGHT_DIR,
    step=15,        # save 1 frame every 15
    max_pairs=40    # set to None to save the whole video
)

### 🎥 How the calibration video was acquired

In this lab, we use the standard calibration setup: the stereo camera is kept fixed, while the chessboard is moved in front of it.

This is the most common and most reliable approach for stereo calibration. By moving the chessboard to different positions, distances, and orientations, we collect a set of views that provides enough geometric variation to estimate the camera parameters accurately. In practice, it is useful to show the chessboard:

- near the center and near the image borders  
- at different distances from the camera  
- with different tilts and orientations  

This allows the calibration algorithm to observe the pattern under a wide range of perspectives.

In principle, it is also possible to do the opposite, **namely to keep the chessboard fixed and move the camera instead**. From a geometric point of view, calibration only requires multiple views of the same known pattern, so both strategies can provide useful information. However, moving the camera is usually less convenient in practice.

The main reason is image quality. When the camera is moved by hand, **the recorded frames are much more likely to contain motion blur**, small vibrations, or unstable framing. This happens because, during the exposure time of the sensor, the camera may still be moving, so the black and white edges of the chessboard are no longer perfectly sharp. Since calibration relies on detecting chessboard corners with high precision, any blur reduces the accuracy of corner localization and makes the final estimation less reliable.

Keeping the stereo camera fixed also has an important advantage for stereo calibration itself. The relative geometry between the left and right cameras remains mechanically stable during the whole acquisition process, and the chessboard can be positioned more easily so that it is clearly visible in both views at the same time. This makes the acquisition more repeatable and easier to control, and usually leads to sharper images and better corner detection.

For these reasons, in this notebook we adopt the standard calibration setup: the camera remains fixed, while the chessboard is moved to different positions, distances, and orientations in front of it.

In [ ]:
CALIB_NAME = "zed_group"
BOARD_COLS = 9                 # inner corners along columns
BOARD_ROWS = 6                 # inner corners along rows
SQUARE_SIZE_MM = 25.0          # physical size of one square in millimeters

print("Camera folder:", MATERIAL_DIR)
print("Left images folder exists:", LEFT_DIR.exists())
print("Right images folder exists:", RIGHT_DIR.exists())


We now load the stereo pairs and visualize a few examples.

A good calibration dataset should include the chessboard:
- near the image center and near the borders
- at different distances from the camera
- with different rotations and tilts
- covering as much of the field of view as possible


In [ ]:

def load_stereo_pairs(left_dir, right_dir):
    left_paths = sorted(list(left_dir.glob("*.png")) + list(left_dir.glob("*.jpg")))
    right_paths = sorted(list(right_dir.glob("*.png")) + list(right_dir.glob("*.jpg")))

    assert len(left_paths) > 0, f"No images found in {left_dir}"
    assert len(right_paths) > 0, f"No images found in {right_dir}"
    assert len(left_paths) == len(right_paths), "Left/right folders must contain the same number of images"

    pairs = []
    for lp, rp in zip(left_paths, right_paths):
        img_l = cv2.imread(str(lp))
        img_r = cv2.imread(str(rp))
        assert img_l is not None, f"Could not read {lp}"
        assert img_r is not None, f"Could not read {rp}"
        pairs.append((lp, rp, img_l, img_r))
    return pairs


pairs = load_stereo_pairs(LEFT_DIR, RIGHT_DIR)
print(f"Loaded {len(pairs)} stereo pairs")

preview_n = min(3, len(pairs))
preview_imgs = []
preview_titles = []
for i in range(preview_n):
    lp, rp, img_l, img_r = pairs[i]
    preview_imgs.extend([img_l, img_r])
    preview_titles.extend([f"Left {i+1}", f"Right {i+1}"])

plot_bgr_list(preview_imgs, preview_titles, figsize=(16, 8))



## 2) Detect chessboard corners

The core step of calibration is the extraction of corresponding **2D image points** from many views of the board.
Each stereo pair contributes:

- the same **3D object points** in the chessboard reference frame
- one set of **2D points in the left image**
- one set of **2D points in the right image**

We first define helper functions for chessboard detection and object-point generation.


In [ ]:

def create_object_points(board_cols, board_rows, square_size_mm):
    objp = np.zeros((board_cols * board_rows, 3), np.float32)
    objp[:, :2] = np.mgrid[0:board_cols, 0:board_rows].T.reshape(-1, 2)
    objp *= square_size_mm
    return objp


def find_checkerboard(gray, board_size):
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE

    if hasattr(cv2, "findChessboardCornersSB"):
        found, corners = cv2.findChessboardCornersSB(gray, board_size, flags)
        if found:
            return True, corners.astype(np.float32)

    found, corners = cv2.findChessboardCorners(
        gray,
        board_size,
        flags | cv2.CALIB_CB_FAST_CHECK,
    )
    if not found:
        return False, None

    criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        30,
        1e-3,
    )
    corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
    return True, corners


The goal of this code is to prepare the two fundamental ingredients needed for camera calibration:

1. the **3D reference points** of the chessboard in its own coordinate system  
2. the corresponding **2D image points** detected in each frame  

In other words, calibration works by matching a set of known 3D points on the chessboard with the 2D corner positions observed in the image.  
The `create_object_points()` function defines the ideal geometric structure of the chessboard, while `find_checkerboard()` detects the corresponding image corners in a grayscale frame.

**`create_object_points()`**

This function creates the set of ideal 3D points associated with the chessboard corners.

Since the chessboard is planar, all points lie on the same plane, so their `z` coordinate is always equal to zero.  
The function first creates a regular grid of corner coordinates, then scales it by the real square size expressed in millimeters.

For example, if the board is `(9, 6)` and the square size is `25 mm`, the generated points correspond to positions such as:

- `(0, 0, 0)`
- `(25, 0, 0)`
- `(50, 0, 0)`
- ...
- `(X, Y, 0)`

These are not the coordinates of the points in the camera reference system. They are the known coordinates of the chessboard corners in the **chessboard reference system**, and they provide the geometric model used during calibration.

**`find_checkerboard()`**

This function receives a grayscale image and tries to detect the inner corners of the chessboard.  
If the pattern is found, it returns the detected corner coordinates. Otherwise, it returns `False`.

The function first defines a set of OpenCV flags to improve robustness under non-ideal lighting conditions.  
Then it tries to detect the chessboard corners using a more recent OpenCV function, and if that is not available or does not succeed, it falls back to the classic detector.  
When the classic detector finds the pattern, the corner positions are further refined at sub-pixel precision.

This makes the function robust and practical for different OpenCV versions.

**[`cv2.findChessboardCornersSB()`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html#gadc5bcb05cb21cf1e50963df26986d7c9)**

This is the newer and generally more robust chessboard detector available in recent OpenCV versions.  
The `SB` version is often better at finding corners under difficult conditions, such as lower contrast, perspective changes, or more challenging views of the pattern.

If the function succeeds, it directly returns the detected chessboard corners, which can then be used for calibration.

**[`cv2.findChessboardCorners()`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html#ga93efa9b0aa890de240ca32b11253dd4a)**

This is the classic OpenCV chessboard detection function.  
It searches for the inner corners of the black-and-white pattern using the board size provided by the user.

**[`cv2.cornerSubPix()`](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html#ga354e0d7c86d0d9da75de9b9701a9a87e)**

This function refines the detected corner locations with **sub-pixel accuracy**.

This step is very important because calibration is highly sensitive to the precision of the corner coordinates.  
Even if the chessboard is detected correctly, the initial corner estimates are usually only approximate at pixel level. `cornerSubPix()` improves them by analyzing the local image intensity around each corner and shifting the point to a more precise location.

This refinement usually leads to more accurate intrinsic and stereo calibration results.

In [ ]:

BOARD_SIZE = (BOARD_COLS, BOARD_ROWS)
objp = create_object_points(BOARD_COLS, BOARD_ROWS, SQUARE_SIZE_MM)

object_points = []
image_points_left = []
image_points_right = []
valid_pairs = []
overlays_left = []
overlays_right = []

for lp, rp, img_l, img_r in pairs:
    gray_l = cv2.cvtColor(img_l, cv2.COLOR_BGR2GRAY)
    gray_r = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)

    found_l, corners_l = find_checkerboard(gray_l, BOARD_SIZE)
    found_r, corners_r = find_checkerboard(gray_r, BOARD_SIZE)

    if found_l and found_r:
        object_points.append(objp.copy())
        image_points_left.append(corners_l)
        image_points_right.append(corners_r)
        valid_pairs.append((lp, rp, img_l, img_r))

        vis_l = img_l.copy()
        vis_r = img_r.copy()
        cv2.drawChessboardCorners(vis_l, BOARD_SIZE, corners_l, True)
        cv2.drawChessboardCorners(vis_r, BOARD_SIZE, corners_r, True)
        overlays_left.append(vis_l)
        overlays_right.append(vis_r)

print(f"Valid stereo pairs for calibration: {len(valid_pairs)} / {len(pairs)}")
assert len(valid_pairs) >= 8, "Try to use at least 8-12 valid pairs for a stable calibration"



This cell processes all the extracted stereo pairs and keeps only the ones that are valid for calibration.

The overall goal is to build the correspondence between:

1. the known **3D chessboard points** in the chessboard reference system  
2. the detected **2D image corners** in the left image  
3. the detected **2D image corners** in the right image  

This is the core information required by stereo calibration.

For each stereo pair, the code first converts the left and right images to grayscale, because chessboard detection works on intensity images rather than color images.  
It then runs `find_checkerboard()` on both views. A stereo pair is considered valid only if the chessboard is successfully detected in **both** the left and the right image.

When this happens, the code stores:

- the same 3D chessboard model in `object_points`
- the detected left image corners in `image_points_left`
- the detected right image corners in `image_points_right`

At the same time, it also saves the original stereo pair inside `valid_pairs`, so that we keep track of which image pairs are actually used in the calibration process.

With the OpenCV function `cv2.drawChessboardCorners()`, the detected corner pattern is shown directly on top of the original frames and it is useful for checking whether the detection is correct. At the end, the code prints how many stereo pairs are valid and compares this number with the total number of available pairs.

**Why do we reject some pairs?** Not all frames are equally useful for calibration. A pair is rejected if the chessboard is not detected in one of the two views.  
This may happen because the chessboard is partially outside the image, too small, too blurred, too tilted, or poorly illuminated. Keeping only valid stereo pairs is important because stereo calibration requires consistent correspondences in both cameras.

**Why do we need multiple valid pairs?** A single stereo pair is not enough to estimate a robust calibration. We need several views of the chessboard at different positions and orientations so that the algorithm can observe enough geometric variation. This is why the code checks that at least a minimum number of valid pairs is available. In practice, around **8–12 valid pairs** can be enough for a basic calibration, but using more well-distributed views usually leads to more stable and accurate results.

**[`cv2.drawChessboardCorners()`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html#ga6a10b0bb120c4907e5eabbcd22319022)**

This function draws the detected chessboard corners on top of an image for visualization.  
It is not used for calibration itself, but it is extremely useful for debugging and for checking whether the detection is correct. If the corners are drawn in the expected order and align well with the chessboard pattern, then the corresponding image pair is likely a good candidate for calibration.



In [ ]:

preview_n = min(3, len(overlays_left))
preview_imgs = []
preview_titles = []
for i in range(preview_n):
    preview_imgs.extend([overlays_left[i], overlays_right[i]])
    preview_titles.extend([f"Detected L{i+1}", f"Detected R{i+1}"])

plot_bgr_list(preview_imgs, preview_titles, figsize=(16, 8))


## 📐 3) Intrinsic calibration of the left and right cameras

Before estimating the stereo geometry, we first calibrate the **left** and **right** cameras independently.

The goal of intrinsic calibration is to estimate the parameters that describe how each camera forms an image.  
These parameters are called **intrinsic parameters** because they depend on the internal characteristics of the camera, such as focal length, principal point, and lens distortion.

In OpenCV, the main outputs of intrinsic calibration are:

- the **camera matrix** `K`
- the **distortion coefficients**
- the pose of the chessboard with respect to the camera in each image (`rvecs`, `tvecs`)

The camera matrix has the form:

$$
K =
\begin{bmatrix}
f_x & 0   & c_x \\
0   & f_y & c_y \\
0   & 0   & 1
\end{bmatrix}
$$

where:

- `fx` and `fy` are the focal lengths expressed in **pixels**
- `(cx, cy)` is the **principal point**, that is, the image point where the optical axis ideally intersects the sensor

In the ideal pinhole camera model, the principal point is often assumed to be close to the image center, so that:

$$
c_x \approx \frac{w}{2}, \qquad c_y \approx \frac{h}{2}
$$


However, in a real camera this is only an approximation, and the calibrated values of `cx` and `cy` may differ slightly from the exact image center.

The distortion coefficients are used to model these lens effects. In particular, they account for:

- **radial distortion**, which typically causes barrel or pincushion effects
- **tangential distortion**, caused by small misalignments between the lens and the image sensor


In [ ]:

image_size = (
    valid_pairs[0][2].shape[1],   # width
    valid_pairs[0][2].shape[0],   # height
)

rms_left, K_left, dist_left, rvecs_left, tvecs_left = cv2.calibrateCamera(
    object_points,
    image_points_left,
    image_size,
    None,
    None,
)

rms_right, K_right, dist_right, rvecs_right, tvecs_right = cv2.calibrateCamera(
    object_points,
    image_points_right,
    image_size,
    None,
    None,
)

print("Left camera matrix:", K_left)
print("Left distortion coefficients:", dist_left.ravel())
print("Right camera matrix:", K_right)
print("Right distortion coefficients:", dist_right.ravel())
print(f"Left RMS:  {rms_left:.4f}")
print(f"Right RMS: {rms_right:.4f}")


**[`cv2.calibrateCamera()`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html#ga3207604e4b1a1758aa66acb6ed5aa65d)**

This function estimates the intrinsic calibration parameters of a single camera from multiple views of a known calibration pattern. It returns the camera model that best explains how the 3D chessboard points are projected onto the image plane.

In this notebook, the function is called twice: once for each lens.

The outputs are:

- **`rms`** — the root mean square reprojection error returned by OpenCV. It gives an overall measure of how well the estimated model fits the observed image points.
- **`K`** — the **camera matrix**, containing the intrinsic parameters of the camera.
- **`dist`** — the **distortion coefficients**, used to model the optical distortion introduced by the lens.
- **`rvecs`** — a list of rotation vectors, one for each calibration image. Each vector describes the orientation of the chessboard with respect to the camera in that view.
- **`tvecs`** — a list of translation vectors, one for each calibration image. Each vector describes the position of the chessboard with respect to the camera in that view.

Together, `rvecs` and `tvecs` represent the per-image pose of the chessboard relative to the camera.  
They are useful because calibration is based on observing the same known pattern from multiple viewpoints.

The most important outputs for the next steps of the notebook are `K` and `dist`, since they describe the intrinsic model of each camera and will later be used in stereo calibration and rectification.

In [ ]:

def reprojection_error(object_points, image_points, rvecs, tvecs, camera_matrix, dist_coeffs):
    errors = []
    for objp, imgp, rvec, tvec in zip(object_points, image_points, rvecs, tvecs):
        projected, _ = cv2.projectPoints(objp, rvec, tvec, camera_matrix, dist_coeffs)
        err = cv2.norm(imgp, projected, cv2.NORM_L2) / len(projected)
        errors.append(float(err))
    return float(np.mean(errors)), errors


mean_err_left, per_view_left = reprojection_error(
    object_points,
    image_points_left,
    rvecs_left,
    tvecs_left,
    K_left,
    dist_left,
)

mean_err_right, per_view_right = reprojection_error(
    object_points,
    image_points_right,
    rvecs_right,
    tvecs_right,
    K_right,
    dist_right,
)

print(f"Mean reprojection error (left):  {mean_err_left:.4f} px")
print(f"Mean reprojection error (right): {mean_err_right:.4f} px")

plt.figure(figsize=(10, 4))
plt.plot(per_view_left, marker="o", label="Left")
plt.plot(per_view_right, marker="s", label="Right")
plt.xlabel("View index")
plt.ylabel("Reprojection error [px]")
plt.title("Per-view reprojection error")
plt.grid(True)
plt.legend()
plt.show()


After calibrating each camera, we need a way to evaluate how well the estimated model explains the observed chessboard corners.  
This is the role of the **reprojection error**.

The idea is the following. For each calibration image, we already know:

- the **3D chessboard points** in the chessboard reference system
- the estimated pose of the chessboard with respect to the camera (`rvec`, `tvec`)
- the estimated intrinsic parameters (`K`) and distortion coefficients

Using all these quantities, we can project the 3D chessboard points back onto the image plane.  
If the calibration is good, these projected points should be very close to the actual detected image corners.


## 🔄 4) Stereo calibration: estimate the Extrinsic Parameters

After calibrating the two cameras independently, we move to the stereo calibration stage.

While the **intrinsic parameters** describe how each camera forms an image, the **extrinsic parameters** describe the position and orientation of a camera with respect to a reference coordinate system.

In general, the extrinsic parameters of a camera are represented by: (i) a **rotation** matrix `R`, (ii) a **translation** vector `T`.
They define a rigid transformation of the form:

$$
\mathbf{x}_c = \mathbf{R}\,\mathbf{X}_w + \mathbf{T}
$$

where:

- $\mathbf{X}_w$ is a 3D point in the world reference system  
- $\mathbf{x}_c$ is the same 3D point expressed in the camera reference system  
- $\mathbf{R}$ is a $3 \times 3$ rotation matrix  
- $\mathbf{T}$ is a $3 \times 1$ translation vector  

These parameters define the rigid transformation that relates a 3D point expressed in one reference system to the camera reference system.

In stereo calibration, however, we are interested in a different type of extrinsic parameters: the fixed rigid transformation between the **left** and **right** cameras of the stereo rig. This transformation tells us how the right camera is positioned and oriented with respect to the left one.

For this reason, stereo calibration estimates, `R` and `T`, are between the two cameras. So, the previous equation become:

$$
\mathbf{x}_{right} = \mathbf{R}\,\mathbf{x}_{left} + \mathbf{T}
$$

These parameters are fundamental because they describe the stereo geometry of the system and are required for the next steps of the pipeline, in particular **rectification**, **disparity computation**, and **3D reconstruction**.

In this step, we use the flag **`cv2.CALIB_FIX_INTRINSIC`**, which means that the intrinsic parameters estimated in the previous section are kept fixed, and only the stereo extrinsic geometry is estimated.

In [ ]:

stereo_criteria = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    100,
    1e-5,
)

stereo_rms, _, _, _, _, R, T, E, F = cv2.stereoCalibrate(
    object_points,
    image_points_left,
    image_points_right,
    K_left,
    dist_left,
    K_right,
    dist_right,
    image_size,
    criteria=stereo_criteria,
    flags=cv2.CALIB_FIX_INTRINSIC,
)

baseline_mm = np.linalg.norm(T)

print("Rotation R:", R)
print("Translation T [mm]:", T.ravel())
print(f"Estimated baseline: {baseline_mm:.2f} mm")
print(f"Stereo RMS: {stereo_rms:.4f}")


**`cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER`**

This expression defines the **termination criterion** used by the optimization algorithm inside `cv2.stereoCalibrate()`.

In calibration, OpenCV estimates the unknown parameters iteratively. This means that it starts from an initial solution and then refines it step by step until a stopping condition is reached.

Here we combine two stopping rules:

- **`cv2.TERM_CRITERIA_MAX_ITER`** — stop after a maximum number of iterations  
- **`cv2.TERM_CRITERIA_EPS`** — stop earlier if the improvement becomes smaller than a given threshold  

here, the criterion is: 
- maximum number of iterations = **100**
- minimum required improvement = **1e-5**

This means that the optimization stops when either: it has performed 100 iterations, or the solution changes by less than `1e-5` between two iterations  

[`cv2.stereoCalibrate()`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html)

This function estimates the **stereo extrinsic parameters** between the left and right cameras.

It takes as input:

- the known **3D object points** of the chessboard  
- the detected **2D image points** in the left images  
- the detected **2D image points** in the right images  
- the intrinsic parameters and distortion coefficients of the left camera  
- the intrinsic parameters and distortion coefficients of the right camera  
- the image size  

The outputs are:

- **`stereo_rms`** — the overall stereo reprojection error returned by OpenCV  
- **`R`** — the rotation matrix between the left and right cameras  
- **`T`** — the translation vector between the left and right cameras  
- **`E`** — the Essential Matrix  
- **`F`** — the Fundamental Matrix  

## 🧭 5) Stereo rectification

Once the stereo extrinsic parameters have been estimated, the next step is **stereo rectification**.

The goal of rectification is to transform the left and right images into a new virtual configuration in which corresponding points lie approximately on the same horizontal image line.  
This is very important because stereo matching becomes much simpler when the search for corresponding points can be restricted to the same row of the two images.

Before rectification, the same 3D point may appear at slightly different vertical positions in the left and right images because of the real geometry of the stereo rig.  
After rectification, this vertical misalignment is minimized, and the correspondence problem becomes essentially one-dimensional.

After rectification, for a corresponding point observed in the left and right images, we ideally have:

$$
y_L' \approx y_R'
$$

This means that the correspondence search can be restricted to the horizontal direction only.
The horizontal displacement between the two rectified points is called **disparity**:

$$
d = x_L' - x_R'
$$

This is the key quantity used in stereo vision to estimate depth. In a simple stereo model, depth is inversely proportional to disparity:

$$
Z = \frac{fB}{d}
$$

where:

- $Z$ is the depth
- $f$ is the focal length
- $B$ is the stereo baseline, the distance between the optical centers of the left and right cameras. In practice, it tells us how far apart the two cameras are. $$\mathrm{baseline} = \|\mathbf{T}\|$$
- $d$ is the disparity

So, rectification is important because it turns the stereo matching problem into a much simpler 1D search along image rows.

In other words, rectification does not change the physical scene, but it geometrically warps the two images so that the stereo pair is easier to process.

In this section, we compute:

- `R1`, `R2`: rectification rotations for the left and right cameras  
- `P1`, `P2`: new projection matrices after rectification  
- `Q`: reprojection matrix used for 3D reconstruction from disparity  
- the remap tables for the left and right images  

The remap tables are especially useful in practice, because once they are computed, they can be applied to every new stereo frame in order to obtain rectified images automatically.

In [ ]:

R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    K_left,
    dist_left,
    K_right,
    dist_right,
    image_size,
    R,
    T,
    flags=cv2.CALIB_ZERO_DISPARITY,
    alpha=0,
)

map1_left, map2_left = cv2.initUndistortRectifyMap(
    K_left,
    dist_left,
    R1,
    P1,
    image_size,
    cv2.CV_32FC1,
)

map1_right, map2_right = cv2.initUndistortRectifyMap(
    K_right,
    dist_right,
    R2,
    P2,
    image_size,
    cv2.CV_32FC1,
)

# Visualize rectification on the first valid stereo pair
_, _, img_l0, img_r0 = valid_pairs[0]
rect_l0 = cv2.remap(img_l0, map1_left, map2_left, cv2.INTER_LINEAR)
rect_r0 = cv2.remap(img_r0, map1_right, map2_right, cv2.INTER_LINEAR)
combined = np.hstack([rect_l0, rect_r0])
combined = draw_horizontal_lines(combined, step=40)

show_bgr(combined, title="Rectified stereo pair with horizontal guide lines", figsize=(16, 6))


: 

**[`cv2.stereoRectify()`](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html)**

This function computes the geometric transformations needed to rectify a calibrated stereo pair.

Starting from:

- the intrinsic parameters of the left and right cameras
- the distortion coefficients
- the stereo extrinsic parameters `R` and `T`

it estimates a new virtual stereo configuration in which corresponding points are aligned as much as possible along the same horizontal lines.

The main outputs are:

- **`R1`, `R2`** — rectification rotations for the left and right cameras  
- **`P1`, `P2`** — new projection matrices for the rectified views  
- **`Q`** — the reprojection matrix used to recover 3D points from disparity  
- **`roi1`, `roi2`** — the valid image regions after rectification  

In practice, `R1` and `R2` define how the two camera views are virtually rotated, while `P1` and `P2` describe the new rectified camera models.  
The matrix `Q` is especially important in stereo vision because it allows us to convert disparity values into 3D coordinates.

**[`cv2.initUndistortRectifyMap()`](https://docs.opencv.org/3.4/da/d54/group__imgproc__transform.html)**

This function builds the pixel-wise remapping tables used to transform an image into its **undistorted and rectified** version.

It combines two operations into a single mapping:

1. **undistortion**, which corrects lens distortion  
2. **rectification**, which aligns the stereo pair geometrically  

The outputs are two maps for each camera, usually called `map1` and `map2`. These maps are not images themselves: they are lookup tables that tell OpenCV where each output pixel should be sampled from in the original image. Once these maps are available, they can be reused for every new frame of the same camera setup, which makes the rectification process efficient in practice.


**[`cv2.remap()`](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html#gab75ef31ce5cdfb5c44b6da5f3b908ea4)**

After the rectification maps have been computed, `cv2.remap()` is used to actually generate the rectified images. Given an input image and the corresponding remap tables, OpenCV creates a new image in which distortion is corrected and the epipolar geometry is aligned.

In [ ]:

calibration_results = {
    "image_size": np.array(image_size, dtype=np.int32),
    "board_size": np.array([BOARD_COLS, BOARD_ROWS], dtype=np.int32),
    "square_size_mm": np.array([SQUARE_SIZE_MM], dtype=np.float32),
    "K_left": K_left,
    "dist_left": dist_left,
    "K_right": K_right,
    "dist_right": dist_right,
    "R": R,
    "T": T,
    "E": E,
    "F": F,
    "R1": R1,
    "R2": R2,
    "P1": P1,
    "P2": P2,
    "Q": Q,
    "roi1": np.array(roi1, dtype=np.int32),
    "roi2": np.array(roi2, dtype=np.int32),
    "map1_left": map1_left,
    "map2_left": map2_left,
    "map1_right": map1_right,
    "map2_right": map2_right,
    "rms_left": np.array([rms_left], dtype=np.float32),
    "rms_right": np.array([rms_right], dtype=np.float32),
    "stereo_rms": np.array([stereo_rms], dtype=np.float32),
    "mean_reprojection_error_left": np.array([mean_err_left], dtype=np.float32),
    "mean_reprojection_error_right": np.array([mean_err_right], dtype=np.float32),
    "per_view_errors_left": np.array(per_view_left, dtype=np.float32),
    "per_view_errors_right": np.array(per_view_right, dtype=np.float32),
}

npz_path = SAVES_DIR / "stereo_calibration.npz"
np.savez_compressed(npz_path, **calibration_results)

summary = {
    "camera_name": CALIB_NAME,
    "image_size": [int(image_size[0]), int(image_size[1])],
    "board_size": [BOARD_COLS, BOARD_ROWS],
    "square_size_mm": SQUARE_SIZE_MM,
    "num_total_pairs": len(pairs),
    "num_valid_pairs": len(valid_pairs),
    "rms_left": float(rms_left),
    "rms_right": float(rms_right),
    "stereo_rms": float(stereo_rms),
    "mean_reprojection_error_left": float(mean_err_left),
    "mean_reprojection_error_right": float(mean_err_right),
    "baseline_mm": float(baseline_mm),
}

summary_path = SAVES_DIR / "stereo_calibration_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print("-", npz_path)
print("-", summary_path)
print("Summary:")
print(json.dumps(summary, indent=2))


At the end of the calibration pipeline, the estimated parameters are saved in two different files.

The first one is an **`.npz` file**, which is a compressed NumPy archive.  
It can store multiple arrays inside a single file, making it very convenient for calibration data such as matrices, vectors, remap tables, and error values. In this case, the `.npz` file contains all the main numerical results of the stereo calibration, including intrinsic parameters, stereo extrinsic parameters, rectification matrices, and reprojection errors.

The second file is a **`.json` summary report**.  
Unlike the `.npz` file, which is mainly designed for numerical data processing in Python, the JSON file is human-readable and provides a compact summary of the most important calibration results, such as image size, board size, number of valid pairs, RMS errors, reprojection errors, and stereo baseline.